# Customer Complaints — Agent Tools + Multi-Turn with Context Provider

This notebook demonstrates:
1. **Tools** — `get_customer_by_id`, `search_customers`, `get_complaints_by_customer`, `get_complaints_by_status`, `register_complaint`, `get_customer_profile` as auto-approved agent tools
2. **Context Provider** — `CustomerProfileProvider` extracts the current customer from conversation and injects their profile tier (platinum/gold/silver/general) into agent instructions
3. **Multi-turn session** — maintains context across turns

## Test Cases
- **A** — Search customer profiles by name "Shweta"
- **B** — Register a complaint for her
- **C** — Get the complaint details

## 1 — Setup: Imports & Path Configuration

In [1]:
import os
import sys
import json
import re
from pathlib import Path
from typing import Any

# Add workspace root to sys.path so customers_complaints is importable
workspace_root = Path.cwd().parent if Path.cwd().name == "use-cases" else Path.cwd()
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from agent_framework._sessions import AgentSession, BaseContextProvider, SessionContext
from agent_framework.azure import AzureOpenAIResponsesClient
from azure.identity import AzureCliCredential

from customers_complaints import (
    init_db,
    get_customer_by_id,
    search_customers,
    get_complaints_by_customer,
    get_complaints_by_status,
    register_complaint,
    get_customer_profile,
)

## 2 — Initialise Database & Seed Data

In [2]:
from customers_complaints.seed import seed

init_db()
seed()  # Safe to call repeatedly — skips if data already exists

Seed complete — 15 customers, profile preferences, and complaints inserted.


## 3 — Context Provider: CustomerProfileProvider

Extracts the current customer's ID from conversation (tool output / user messages) and injects their profile tier into agent instructions before each run.

In [3]:
class CustomerProfileProvider(BaseContextProvider):
    """Tracks the current customer from conversation and injects profile-tier context."""

    def __init__(self) -> None:
        super().__init__(source_id="customer-profile-provider")
        self.current_customer_id: str | None = None
        self.current_customer_name: str | None = None
        self.current_customer_type: str | None = None

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Inject profile-tier context into agent instructions."""
        if self.current_customer_id and self.current_customer_type:
            context.instructions.append(
                f"Current customer context: {self.current_customer_name} "
                f"(ID: {self.current_customer_id}, Tier: {self.current_customer_type}). "
                f"When registering complaints for this customer, use their customer_id '{self.current_customer_id}'. "
                f"For platinum-tier customers, the system automatically sets priority to High if not specified."
            )
        else:
            context.instructions.append(
                "No customer is currently selected. Help the user find a customer first."
            )

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Extract customer_id from tool output or conversation to track context."""
        for msg in context.input_messages:
            text = msg.text if hasattr(msg, "text") else str(msg)
            if not isinstance(text, str):
                continue

            # Look for customer_id pattern in the message / tool output
            match = re.search(r'CUST\d{5}', text)
            if match:
                found_id = match.group(0)
                if found_id != self.current_customer_id:
                    self.current_customer_id = found_id
                    # Look up profile
                    profile_json = get_customer_profile(found_id)
                    try:
                        profile_data = json.loads(profile_json)
                        self.current_customer_name = profile_data.get("customer_name")
                        self.current_customer_type = profile_data.get("customer_type")
                    except (json.JSONDecodeError, TypeError):
                        pass
                    print(f"[ProfileProvider] Tracked customer: {self.current_customer_name} ({self.current_customer_id}) — {self.current_customer_type}")

## 4 — Agent Setup

In [4]:
credential = AzureCliCredential()
client = AzureOpenAIResponsesClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    deployment_name=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=credential,
)

profile_provider = CustomerProfileProvider()

agent = client.as_agent(
    name="ComplaintsAgent",
    instructions=(
        "You are an IT helpdesk assistant that manages customer records and IT complaints. "
        "Use the available tools to search customers, view complaints, register new complaints, "
        "and retrieve customer profile information. Always confirm actions with the user. "
        "When a customer is identified, use their customer_id for subsequent operations."
    ),
    tools=[
        get_customer_by_id,
        search_customers,
        get_complaints_by_customer,
        get_complaints_by_status,
        register_complaint,
        get_customer_profile,
    ],
    context_providers=[profile_provider],
)

## 5 — Create Session

In [5]:
session = agent.create_session()

## Test Case A — Search customer profiles by name "Shweta"

In [6]:
result = await agent.run("Search for a customer named Shweta", session=session)
print(f"Agent: {result}\n")

Agent: I found a customer matching the name "Shweta":

- **Name:** Shweta Iyer  
- **Customer ID:** CUST10001  
- **Address:** 42, MG Road, Indiranagar, Bangalore 560038  
- **Email:** shweta.iyer@email.in  
- **Phone:** +91-80-98765432  
- **Status:** Active  
- **Remarks:** Key enterprise account  

Would you like to proceed with this customer?



## Test Case B — Register a complaint for Shweta

In [7]:
result = await agent.run(
    "Register an IT complaint for Shweta — her laptop is not connecting to the corporate VPN since this morning.",
    session=session,
)
print(f"Agent: {result}\n")

Agent: The IT complaint for Shweta Iyer regarding her laptop not connecting to the corporate VPN has been successfully registered:

- **Complaint ID:** COMP10066  
- **Date:** April 14, 2026  
- **Description:** Laptop is not connecting to the corporate VPN since this morning.  
- **Priority:** High  
- **Status:** Open  

Is there anything else you'd like me to assist with?



## Test Case C — Get complaint details for Shweta

In [8]:
result = await agent.run("Show me all the complaint details for this customer.", session=session)
print(f"Agent: {result}\n")

Agent: Here are the complaint details for Shweta Iyer:

1. **Complaint ID:** COMP10066  
   - **Date:** April 14, 2026  
   - **Description:** Laptop is not connecting to the corporate VPN since this morning.  
   - **Priority:** High  
   - **Status:** Open  

2. **Complaint ID:** COMP10003  
   - **Date:** March 19, 2026  
   - **Description:** Mouse and keyboard lag on remote desktop session  
   - **Priority:** High  
   - **Status:** Closed  

3. **Complaint ID:** COMP10002  
   - **Date:** March 9, 2026  
   - **Description:** Cannot access shared network drive `\\fileserver\projects`  
   - **Priority:** High  
   - **Status:** Closed  

4. **Complaint ID:** COMP10001  
   - **Date:** March 5, 2026  
   - **Description:** Microsoft Teams screen sharing not working  
   - **Priority:** High  
   - **Status:** In Progress  

5. **Complaint ID:** COMP10004  
   - **Date:** March 3, 2026  
   - **Description:** Slow internet speed affecting video conferencing  
   - **Priority:** Hi

## Verify — Check tracked profile context

In [9]:
print(f"[Context] Customer ID  : {profile_provider.current_customer_id}")
print(f"[Context] Customer Name: {profile_provider.current_customer_name}")
print(f"[Context] Customer Type: {profile_provider.current_customer_type}")

[Context] Customer ID  : None
[Context] Customer Name: None
[Context] Customer Type: None
